# ルーブリック（観点別の採点基準）に基づいて作文を採点する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/30_language_education/rubric_scoring.ipynb)

NLP クックブック 30-08 ／ 言語教育・リーダビリティ・自動採点

総合点 1 スカラーだけでなく、「内容・論点」「構成・論理展開」「表現・語彙」「文法・表記」といった複数観点のルーブリックに基づいて作文を採点し、観点ごとの強みと改善点のフィードバックまで生成したい。ASAP データセットのうち 4 観点のアナリティック採点が付いた課題 7 を用いて、観点別採点と総合点予測を比較する。

In [1]:
!pip install -q datasets sentence-transformers wordfreq

In [2]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from wordfreq import zipf_frequency

df = load_dataset("nlpatunt/D_ASAP-AES", split="train").to_pandas()
es7 = df[df["essay_set"] == 7].dropna(subset=["domain1_score", "rater1_trait1", "rater2_trait1"]).reset_index(drop=True)

# ASAP 課題 7 の 4 観点ルーブリック（各採点者 0〜3 点、2 人の合計 0〜6 点。4 観点の合計が domain1_score = 0〜24 点）
TRAITS = [
    ("trait1", "内容・論点 (Ideas)"),
    ("trait2", "構成・展開 (Organization)"),
    ("trait3", "表現・語彙 (Style)"),
    ("trait4", "文法・表記 (Conventions)"),
]
for k, _ in TRAITS:
    es7[k] = (es7[f"rater1_{k}"] + es7[f"rater2_{k}"]).astype(int)
es7["total"] = es7["domain1_score"].astype(int)
print(f"課題 7 の作文数: {len(es7)} 件 / 総合点の範囲: {es7['total'].min()}〜{es7['total'].max()}")

課題 7 の作文数: 1098 件 / 総合点の範囲: 2〜24


In [3]:
# 観点ごとの特徴量を設計する: 分量・段落/接続表現・語彙の豊かさと頻度・綴り/句読点の正確さ
CONNECTIVES = {
    "however", "therefore", "moreover", "furthermore", "although", "because", "since",
    "while", "then", "next", "finally", "first", "second", "consequently", "meanwhile", "instead",
}

def rubric_features(text):
    words = re.findall(r"[A-Za-z']+", text)
    lower = [w.lower() for w in words]
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    paras = [p.strip() for p in text.splitlines() if p.strip()]
    nw = max(len(lower), 1)
    zipfs = [zipf_frequency(w, "en") for w in lower if w.isalpha()]
    return {
        "log_words": np.log1p(len(words)),
        "n_sents": len(sents),
        "avg_sent_len": len(words) / max(len(sents), 1),
        "n_paras": len(paras),
        "conn_rate": sum(w in CONNECTIVES for w in lower) / max(len(sents), 1),
        "ttr": len(set(lower)) / nw,
        "avg_word_len": float(np.mean([len(w) for w in lower] or [0.0])),
        "rare_rate": sum(0 < z < 4.5 for z in zipfs) / max(len(zipfs), 1),
        "oov_rate": sum(z == 0 for z in zipfs) / max(len(zipfs), 1),
        "comma_rate": text.count(",") / max(len(sents), 1),
        "cap_start_rate": sum(bool(re.match(r"^[A-Z@]", s)) for s in sents) / max(len(sents), 1),
    }

feat_df = pd.DataFrame([rubric_features(t) for t in es7["essay"]])
corr_table = pd.DataFrame({name: [feat_df[c].corr(es7[k]) for c in feat_df.columns] for k, name in TRAITS}, index=feat_df.columns)
print("=== 各特徴量と人手評点（観点別）の相関係数 ===")
print(corr_table.round(3).to_string())

=== 各特徴量と人手評点（観点別）の相関係数 ===
                内容・論点 (Ideas)  構成・展開 (Organization)  表現・語彙 (Style)  文法・表記 (Conventions)
log_words               0.640                 0.667          0.672                0.581
n_sents                 0.557                 0.592          0.625                0.602
avg_sent_len           -0.129                -0.171         -0.252               -0.377
n_paras                 0.042                 0.016          0.078                0.049
conn_rate              -0.030                -0.072         -0.156               -0.232
ttr                    -0.408                -0.417         -0.346               -0.328
avg_word_len           -0.065                 0.098          0.254                0.272
rare_rate               0.169                 0.188          0.260                0.152
oov_rate               -0.137                -0.158         -0.169               -0.287
comma_rate              0.065                 0.114          0.204                0.171
cap_

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

embedder = SentenceTransformer("all-MiniLM-L6-v2")
E = embedder.encode(es7["essay"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
X = feat_df.to_numpy()
cv = KFold(5, shuffle=True, random_state=0)
alphas = np.logspace(-1, 3, 9)
qwk = lambda yt, yp: cohen_kappa_score(yt, yp, weights="quadratic")

oof_len = {k: np.empty(len(es7), dtype=int) for k, _ in TRAITS}
oof_feat = {k: np.empty(len(es7), dtype=int) for k, _ in TRAITS}
oof_all = {k: np.empty(len(es7), dtype=int) for k, _ in TRAITS}
oof_all_cont = {k: np.empty(len(es7)) for k, _ in TRAITS}
oof_direct = np.empty(len(es7), dtype=int)

for tr, te in cv.split(X):
    scaler = StandardScaler().fit(X[tr])
    Xtr_s, Xte_s = scaler.transform(X[tr]), scaler.transform(X[te])
    Xtr_all, Xte_all = np.hstack([Xtr_s, E[tr]]), np.hstack([Xte_s, E[te]])
    for k, _ in TRAITS:
        y_tr = es7[k].to_numpy()[tr]
        oof_len[k][te] = np.clip(np.rint(RidgeCV(alphas=alphas).fit(Xtr_s[:, :1], y_tr).predict(Xte_s[:, :1])), 0, 6).astype(int)
        oof_feat[k][te] = np.clip(np.rint(RidgeCV(alphas=alphas).fit(Xtr_s, y_tr).predict(Xte_s)), 0, 6).astype(int)
        pred = RidgeCV(alphas=alphas).fit(Xtr_all, y_tr).predict(Xte_all)
        oof_all_cont[k][te] = pred
        oof_all[k][te] = np.clip(np.rint(pred), 0, 6).astype(int)
    oof_direct[te] = np.clip(np.rint(RidgeCV(alphas=alphas).fit(Xtr_all, es7["total"].to_numpy()[tr]).predict(Xte_all)), 0, 24).astype(int)

rows = []
for k, name in TRAITS:
    rows.append({
        "観点": name,
        "語数のみ": qwk(es7[k], oof_len[k]),
        "設計特徴量": qwk(es7[k], oof_feat[k]),
        "特徴量+埋め込み": qwk(es7[k], oof_all[k]),
        "採点者間QWK": qwk(es7[f"rater1_{k}"].astype(int), es7[f"rater2_{k}"].astype(int)),
    })
print("\n=== 観点別の採点精度 (QWK) ===")
print(pd.DataFrame(rows).set_index("観点").round(3).to_string())

sum_pred = np.clip(np.rint(sum(oof_all_cont[k] for k, _ in TRAITS)), 0, 24).astype(int)
human_tot = qwk(es7["rater1_domain1"].astype(int), es7["rater2_domain1"].astype(int))
print(f"\n総合点 QWK: 直接予測={qwk(es7['total'], oof_direct):.3f} / 4 観点合算={qwk(es7['total'], sum_pred):.3f} (参考: 採点者間={human_tot:.3f})")


=== 観点別の採点精度 (QWK) ===
                       語数のみ  設計特徴量  特徴量+埋め込み  採点者間QWK
観点                                                   
内容・論点 (Ideas)         0.554  0.638     0.757    0.683
構成・展開 (Organization)  0.589  0.628     0.683    0.587
表現・語彙 (Style)         0.588  0.707     0.712    0.533
文法・表記 (Conventions)   0.489  0.660     0.685    0.565

総合点 QWK: 直接予測=0.814 / 4 観点合算=0.813 (参考: 採点者間=0.717)


In [5]:
# 予測された観点別スコアに基づくフィードバック生成（強みと改善すべき観点を提示）
es7["pred_sum"] = sum_pred
for k, _ in TRAITS:
    es7[f"pred_{k}"] = oof_all[k]
es7["gap"] = es7["pred_trait1"] - es7["pred_trait4"]

ADVICE = {
    "trait1": ("具体的な体験や描写がよく書けています。", "話題に沿った具体例や背景の説明をもう少し補いましょう。"),
    "trait2": ("話の流れが整理され、まとまりがあります。", "場面の切り替わりや接続表現を意識して段落構成を整えましょう。"),
    "trait3": ("状況が伝わる表現や語彙の工夫が見られます。", "同じ語の繰り返しを避け、具体的な形容詞・動詞を増やしましょう。"),
    "trait4": ("句読点や綴り・文法の誤りが少なく読みやすいです。", "文の区切り（ピリオド）やスペル・大文字小文字を見直しましょう。"),
}

samples = pd.concat([es7.nlargest(1, "gap"), es7.nsmallest(1, "gap"), es7.nlargest(1, "pred_sum")])
for _, row in samples.iterrows():
    breakdown = ", ".join(f"{name.split()[0]}={row[f'pred_{k}']}/6(正解{row[k]})" for k, name in TRAITS)
    best_k = max(TRAITS, key=lambda x: row[f"pred_{x[0]}"])[0]
    weak_k = min(TRAITS, key=lambda x: row[f"pred_{x[0]}"])[0]
    print(f"\n[総合予測 {row['pred_sum']}/24 (正解 {row['total']})] {breakdown}")
    print("  冒頭:", row["essay"][:90].replace("\n", " "))
    print("  強み:", ADVICE[best_k][0])
    print("  改善:", "すべての観点で十分な基準を満たしています。" if row[f"pred_{weak_k}"] >= 5 else ADVICE[weak_k][1])


[総合予測 12/24 (正解 13)] 内容・論点=4/6(正解4), 構成・展開=4/6(正解4), 表現・語彙=3/6(正解3), 文法・表記=1/6(正解2)
  冒頭: One day me and my mom dad and sister were going to @LOCATION1 to go to unversual it is a a
  強み: 具体的な体験や描写がよく書けています。
  改善: 文の区切り（ピリオド）やスペル・大文字小文字を見直しましょう。

[総合予測 9/24 (正解 8)] 内容・論点=0/6(正解1), 構成・展開=2/6(正解2), 表現・語彙=3/6(正解2), 文法・表記=4/6(正解3)
  冒頭: Patience is one of the most important things in life. Patience is being understanding and 
  強み: 句読点や綴り・文法の誤りが少なく読みやすいです。
  改善: 話題に沿った具体例や背景の説明をもう少し補いましょう。

[総合予測 24/24 (正解 24)] 内容・論点=6/6(正解6), 構成・展開=6/6(正解6), 表現・語彙=6/6(正解6), 文法・表記=6/6(正解6)
  冒頭: Here I was on @CAPS1 third day of volley ball tryouts. I was sure I was going to be on the
  強み: 具体的な体験や描写がよく書けています。
  改善: すべての観点で十分な基準を満たしています。


- 観点別に予測すると、単なる「語数だけ」のベースラインに対して、文埋め込みを加えたモデルは「内容・論点 (Ideas)」で特に大きく伸び (0.554 → 0.757)、綴りミス率 (`oov_rate`) や平均文長（句読点の欠落で長くなる）などを含む設計特徴量は「表現・語彙 (Style)」(0.588 → 0.707) と「文法・表記 (Conventions)」(0.489 → 0.660) を大きく改善する。
- 4 観点の連続予測を合算した総合点 QWK (0.813) は、総合点を直接予測した QWK (0.814) とほぼ同等でありながら、「内容は書けているが句読点・綴りが弱い」「文法は正確だが具体例が不足している」といった学習者ごとの指導ポイントを定量的に示せる。
- 前のレシピ「作文を自動で採点する」と同様、モデルが比べている正解は 2 人の採点者の合計点（0〜6 点・0〜24 点）であるため、1 人の採点者同士の一致度より見かけ上の QWK が高く出る点に注意する。また、文脈の誤りや論理の一貫性を細かく指摘するには、文法誤り訂正と文章校正の章の「訂正にフィードバックコメントを付ける」のような生成モデルとの併用が有効。